# Training a rat model in RODENT: a worked example

**Aim.** Train a PyTorch model to control the virtual rat in a [RODENT](https://rodent-tau.vercel.app/) experiment, measure how well it learned, and save the trained model and its runs to the website so the rest of the project can use them.

**The example.** In the light/dark box a real rat is placed in the bright compartment and, given time, usually moves into the dark one. Here a model starts with no idea what to do, and learns to leave the light and reach the dark room through trial and error. Other goals (avoid light, follow an odour, follow a sound) are one setting away in step 5.

**How the notebook is organised**

| | Step | You change it? |
|---|---|---|
| 1 | Setup: install RODENT, download the model file | |
| 2 | Sign in to the website | |
| 3 | Choose the experiment | ✏️ |
| 4 | Explore the experiment: what is in the arena, what the rat senses, how the built-in rat behaves | |
| 5 | Define the task: the goal, where the rat starts, how long it trains | ✏️ |
| 6 | The model (from `rat_model.py`) | ✏️ only to bring your own |
| 7 | Train | |
| 8 | Results: learning curve, before and after, any episode, trained rat vs built-in rat | |
| 9 | Save to the website, and run it live there | ✏️ model name |
| 10 | Reuse the model in another experiment | ✏️ optional |

**Before you start:** click the 🔑 key icon on the left (**Secrets**), add `RODENT_EMAIL` and `RODENT_PASSWORD` with your website login, and turn on **Notebook access** for both. Then **Runtime → Run all**. Training takes about 5 to 10 minutes.

### Words used in this notebook

- **Episode:** one attempt. The rat is placed in the arena and gets a fixed number of steps (10 steps = 1 second of rat time).
- **Observation:** what the rat senses before each step: 31 numbers between -1 and 1 (its position and heading, the distance to walls in 8 directions, and how strong the light, sound and odour are, on which side, and where they come from).
- **Action:** the move the model makes on each step, one of 5: stay, forward, turn left, turn right, run.
- **Reward:** a number after each step that scores it against the goal. Training changes the model so that it collects more reward.
- **Round:** a batch of episodes played with the current model, after which the model learns from them.

## 1. Setup

In [ ]:
# RODENT's Python package, with the extras for training (Gymnasium, pandas, matplotlib).
# PyTorch is already installed in Colab.
%pip install --quiet "rodent-sdk[train]==0.4.4" onnx

In [ ]:
import math
import os
import urllib.request

import matplotlib.pyplot as plt
import pandas as pd
import torch
import onnx
import rodent_sdk as rodent

# The model is kept in its own file, rat_model.py, so it can be read, changed and shared on its own.
# Download the example the first time. To use your own model instead, upload your rat_model.py with the
# folder icon on the left before running this cell (see step 6 for what the file must contain).
MODEL_FILE_URL = "https://raw.githubusercontent.com/HuzaifaMajeed/rodent-sdk-releases/main/notebooks/rat_model.py"
if not os.path.exists("rat_model.py"):
    urllib.request.urlretrieve(MODEL_FILE_URL, "rat_model.py")
from rat_model import RatModel

print("RODENT", rodent.__version__, "| PyTorch", torch.__version__)

## 2. Sign in

In [ ]:
# Reads RODENT_EMAIL and RODENT_PASSWORD from Colab's Secrets (or asks for them).
client = rodent.login()

print("Your projects:")
for project in client.projects():
    print(" -", project.name)

## 3. Choose the experiment ✏️

On the website, open your experiment, click **Use in Python** and paste the code into `EXPERIMENT_CODE`. Or type the project and experiment names.

The trained model will be saved to this experiment's **project**, so every experiment in the project can use it (step 10).

**No light/dark box experiment yet?** Type a name into `NEW_PROJECT` and this cell creates a project from the Light/Dark Box template. Its experiment is called *name* + ` baseline`.

In [ ]:
EXPERIMENT_CODE = ""  #@param {type:"string"}
PROJECT = ""  #@param {type:"string"}
EXPERIMENT = ""  #@param {type:"string"}
NEW_PROJECT = ""  #@param {type:"string"}

if NEW_PROJECT:
    if NEW_PROJECT not in [p.name for p in client.projects()]:
        client.create_project(NEW_PROJECT, template="light_dark_box")
    PROJECT, EXPERIMENT = NEW_PROJECT, NEW_PROJECT + " baseline"
elif EXPERIMENT_CODE:
    PROJECT, EXPERIMENT = EXPERIMENT_CODE.split("/")

project = client.project(PROJECT)
exp = project.experiment(EXPERIMENT)
print(f"Experiment: {exp.name}  (project: {project.name}, seed {exp.seed})")

## 4. Explore the experiment

Before training anything, look at what the rat will be working with. The names printed here are the ones to use in step 5.

In [ ]:
# Regions are named areas of the arena; doors can be open or closed; stimuli are light, sound and
# odour sources. A source only works if it is *enabled* (present in the arena) and *active* (switched on).
print("Regions:", exp.regions)
print("Doors:  ", exp.doors)
print("Stimuli:")
for source in exp.config["stimuli"]:
    enabled = source.get("enabled", True)
    state = "on" if enabled and source.get("initially_active") else ("off" if enabled else "not enabled")
    print(f"  {source['id']:<16} {source['type']:<6} {state:<12} at {source['position']}, radius {source['radius']} cm")

**What the rat can sense.** Each map shows how strongly the rat would sense that stimulus at each spot of the arena (brighter = stronger). Lamps hang overhead and light a pool on the floor below them; a wall shades the floor just behind it, and a covered room under a lamp stays dark. Sound and odour get weaker through walls. Odour is a gas that spreads over time, so its map is taken after 300 steps.

In [ ]:
kinds = ["light", "sound", "odour"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for kind, ax in zip(kinds, axes):
    exp.stimulus_map(kind, after_steps=300 if kind == "odour" else 0, ax=ax)
    ax.set_title(kind)
plt.tight_layout()
plt.show()

**How the built-in rat behaves.** RODENT's own rat follows a statistical model of real rat behaviour. Its run is the baseline the trained rat is compared with in step 8.

In [ ]:
builtin_run = exp.run(steps=300)
builtin_run.plot()
plt.show()
builtin_run.region_table().round(1)

## 5. Define the task ✏️

**`TASK`** is what you want the rat to learn. Each task is a reward (scored after every step) and a measure (how you judge whether it worked):

| `TASK` | What the rat should learn | Reward after each step | Measure |
|---|---|---|---|
| `reach_region` | get to the region `GOAL` | +1 inside `GOAL`; before that, a small minus that shrinks as it gets closer | % of episodes that end in `GOAL` |
| `avoid_light` | stay out of the light | minus how bright it is (0 in the dark) | % of the time spent in the dark |
| `follow_odour` | find the odour source | how strong the odour is | odour strength over the last 20% of each episode |
| `follow_sound` | find the sound source | how loud the sound is | loudness over the last 20% of each episode |

**Tips for the other tasks**

- `follow_odour` and `follow_sound` need a source of that kind that is **enabled** and **active**. On the website, select the source in the arena editor, tick **Enabled** and **Active at start**, and save. In the light/dark box, the odour source sits in the dark room.
- Odour spreads slowly, so give it longer episodes (`STEPS_PER_EPISODE = 300`).
- Sound is only heard inside its radius (18 cm in the templates). Make the radius larger on the website (50 cm works) or start the rat near it.

**The other settings**

| Setting | What it does |
|---|---|
| `GOAL` | the region for `reach_region` |
| `START` | where the rat starts each episode, `x, y` in cm (facing a random way). Empty = a random spot each time. `20, 50` is next to the lamp in the light/dark box |
| `STEPS_PER_EPISODE` | length of each episode (150 steps = 15 seconds) |
| `ROUNDS` | how long to train. Each round is `EPISODES_PER_ROUND` episodes followed by one learning step |
| `SEED` | makes the training repeatable: the same seed gives the same result |

In [ ]:
TASK = "reach_region"  #@param ["reach_region", "avoid_light", "follow_odour", "follow_sound"]
GOAL = "dark_room"  #@param {type:"string"}
START = "20, 50"  #@param {type:"string"}
STEPS_PER_EPISODE = 150  #@param {type:"integer"}
ROUNDS = 80  #@param {type:"integer"}
EPISODES_PER_ROUND = 8  #@param {type:"integer"}
SEED = 0  #@param {type:"integer"}

start = {"random": True}                        # random heading
if START:
    start["position"] = [float(v) for v in START.split(",")]


# ---- the four tasks: a reward for every step, and a measure for every episode -------------------
# step has: region, position (x, y), hit_wall, moved (cm), light, sound, odour (each 0 to 1).
# An episode's measure gets the list of every step's info (region, position, senses, ...).

def goal_centre():
    x, y, width, height = next(r["rect"] for r in exp.config["arena"]["regions"] if r["id"] == GOAL)
    return (x + width / 2, y + height / 2)


def reach_region_reward(step):
    if step.region == GOAL:
        return 1.0
    # "Warmer, colder": closer to the middle of the goal is less negative. Without this hint the
    # rat would only learn anything after stumbling into the goal by chance.
    return -math.dist(step.position, GOAL_CENTRE) / 100


def last_fifth(infos):
    return infos[-max(1, len(infos) // 5):]


TASKS = {
    "reach_region": dict(
        reward=reach_region_reward,
        measure=lambda infos: 100.0 * (infos[-1]["region"] == GOAL),
        measure_name=f"% of episodes ending in {GOAL}",
        description=f"Reach the region {GOAL}"),
    "avoid_light": dict(
        reward=lambda step: -step.light,
        measure=lambda infos: 100.0 * sum(i["senses"]["light"] < 0.05 for i in infos) / len(infos),
        measure_name="% of time in the dark",
        description="Avoid the light"),
    "follow_odour": dict(
        reward=lambda step: step.odour,
        measure=lambda infos: sum(i["senses"]["odour"] for i in last_fifth(infos)) / len(last_fifth(infos)),
        measure_name="odour strength near the end (0 to 1)",
        description="Follow the odour to its source"),
    "follow_sound": dict(
        reward=lambda step: step.sound,
        measure=lambda infos: sum(i["senses"]["sound"] for i in last_fifth(infos)) / len(last_fifth(infos)),
        measure_name="loudness near the end (0 to 1)",
        description="Follow the sound to its source"),
}

# ---- check the task makes sense in this experiment -----------------------------------------------
if TASK == "reach_region":
    if GOAL not in exp.regions:
        raise ValueError(f"No region called {GOAL!r}. Choose one of {exp.regions}")
    GOAL_CENTRE = goal_centre()
if TASK in ("follow_odour", "follow_sound"):
    kind = TASK.split("_")[1]
    working = [s["id"] for s in exp.config["stimuli"]
               if s["type"] == kind and s.get("enabled", True) and s.get("initially_active")]
    if not working:
        raise ValueError(f"{TASK} needs an {kind} source that is enabled and active. On the website, select "
                         f"the source, tick Enabled and Active at start, save, and run the notebook again.")

reward = TASKS[TASK]["reward"]
measure = TASKS[TASK]["measure"]
MEASURE_NAME = TASKS[TASK]["measure_name"]
print("Task:", TASKS[TASK]["description"], "| measured as:", MEASURE_NAME)

## 6. The model

The model is in its own file, **`rat_model.py`** (open it from the folder icon on the left to read it; every part is explained there). In short:

- **What it is:** a PPO *actor-critic*. The **actor** is a small neural network that scores the 5 moves from the 31 observations; the rat picks moves in proportion to those scores, so it keeps exploring while it learns. The **critic** is a second network that estimates how much reward is still to come from where the rat is.
- **How it learns:** after each round, every move is judged by whether things went better or worse than the critic expected after it (the *advantage*). Better moves become more likely. PPO limits how much the actor can change in one round, which keeps training stable.

**Bringing your own model:** replace `rat_model.py` with your own file that defines `RatModel` with:

- `forward(observations)`: 31 numbers in, a score for each of the 5 moves out;
- `act(observation, best=False)`: returns the move (0 to 4). RODENT also calls this when the model drives the rat in a run;
- `learn(episodes)`: improves from a list of finished episodes (each has `observations`, `actions`, `rewards`, `final_observation`, `terminated`) and returns a dict of numbers to print.

In [ ]:
torch.manual_seed(SEED)
model = RatModel()
print(model.describe())
model

## 7. Train

First, a training environment: the experiment, the task's reward, the start and the episode length together. Every training episode is logged in the folder `rodent-training`, so any of them can be rebuilt and watched later.

A second environment with no log is used only for testing the model, before and after training, on the same 20 episodes (same starts, same headings, same random choices), so the two numbers can be compared fairly.

In [ ]:
env = exp.training_env(max_steps=STEPS_PER_EPISODE, reward=reward, actions="discrete",
                       start=start, log=f"rodent-training/{TASK}")
test_env = exp.training_env(max_steps=STEPS_PER_EPISODE, reward=reward, actions="discrete", start=start)
history = []                                     # one row per training round


def play(environment, best=False, seed=None):
    """One episode with the current model. Returns what the model needs to learn, plus every step's info."""
    observation, info = environment.reset(seed=seed)
    episode = {"observations": [], "actions": [], "rewards": []}
    infos, done = [], False
    while not done:
        action = model.act(observation, best=best)
        episode["observations"].append(list(observation))
        episode["actions"].append(action)
        observation, r, terminated, truncated, info = environment.step(action)
        episode["rewards"].append(r)
        infos.append(info)
        done = terminated or truncated
    episode["final_observation"] = list(observation)
    episode["terminated"] = terminated
    return episode, infos


def test(n=20):
    """The model's average measure on n fixed test episodes. The model picks its moves the way it does
    in training; seeding PyTorch makes those choices, and so the result, the same every time."""
    with torch.random.fork_rng():
        torch.manual_seed(SEED)
        return sum(measure(play(test_env, seed=10_000 + i)[1]) for i in range(n)) / n


before = test()
print(f"Before training: {MEASURE_NAME} = {before:.2f}")

Now the training itself: `ROUNDS` rounds of `EPISODES_PER_ROUND` episodes, each followed by one learning step. It prints progress every 5 rounds. If the measure is still going up at the end, run this cell again: it carries on from where it stopped.

In [ ]:
for _ in range(ROUNDS):
    episodes, measures = [], []
    for _ in range(EPISODES_PER_ROUND):
        episode, infos = play(env)
        episodes.append(episode)
        measures.append(measure(infos))
    losses = model.learn(episodes)
    history.append({"round": len(history) + 1,
                    "mean_reward": sum(sum(e["rewards"]) for e in episodes) / len(episodes),
                    "measure": sum(measures) / len(measures), **losses})
    if len(history) % 5 == 0:
        row = history[-1]
        print(f"round {row['round']:>3}: mean reward {row['mean_reward']:7.1f} | {MEASURE_NAME}: {row['measure']:.2f}")

history_table = pd.DataFrame(history)

## 8. Results

### Learning curve

Left: the mean total reward per episode in each round. Right: the task's measure. Both should rise and then level off.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4))
left.plot(history_table["round"], history_table["mean_reward"])
left.set(xlabel="round", ylabel="mean reward per episode", title="Reward")
right.plot(history_table["round"], history_table["measure"], color="tab:green")
right.set(xlabel="round", ylabel=MEASURE_NAME, title="Measure")
plt.tight_layout()
plt.show()

# RODENT's own view of the log: reward per episode, and how each episode's time was split between regions.
env.log.plot()
plt.show()

### Before and after training (the same 20 test episodes)

Read the number against the task: for `avoid_light`, for example, the rat does not have to reach the dark room. The lamp only lights part of the bright room, and a rat that waits in an unlit corner of it scores just as well.

In [ ]:
after = test()
print(f"{MEASURE_NAME}")
print(f"  before training: {before:.2f}")
print(f"  after training:  {after:.2f}")

### Watch any episode

Every training episode can be rebuilt exactly from the log, without the model. Set `EPISODE` (0 = the last one) and `AT_STEP` to see what the rat sensed at that moment.

In [ ]:
EPISODE = 0  #@param {type:"integer"}
AT_STEP = 50  #@param {type:"integer"}

last_episode = env.log.next_episode - 1
episode = env.log.episode(EPISODE or last_episode)
episode.plot()
plt.show()

moment = episode.at(AT_STEP)
x, y = moment["position"]
print(f"Episode {episode.number}: total reward {episode.total_reward:.1f}")
print(f"At step {AT_STEP}: in {moment['region']} at ({x:.0f}, {y:.0f}) cm, senses {moment['senses']}")
episode.animate()

The first, a middle and the last training episode side by side:

In [ ]:
env.log.compare([1, last_episode // 2, last_episode])
plt.show()

### The trained rat next to the built-in rat

The trained model now drives the rat in an ordinary run of the experiment, twice as long as a training episode. RODENT asks the model for each move through its `act` method, exactly as in training, and seeds it from the experiment's seed, so the same run comes out every time. `episode_length` tells the model how long its training episodes were, so its sense of time means the same as when it learned.

In [ ]:
model_run = exp.run(steps=2 * STEPS_PER_EPISODE, model=model, actions="discrete",
                    start=start.get("position"), episode_length=STEPS_PER_EPISODE)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 5))
builtin_run.plot(ax=left)
left.set_title("Built-in rat")
model_run.plot(ax=right)
right.set_title("Trained rat")
plt.show()

rodent.compare(built_in_rat=builtin_run, trained_rat=model_run)

### Summary

In [ ]:
print(f"Task:            {TASKS[TASK]['description']} ({exp.name})")
print(f"Training:        {len(history)} rounds, {len(history) * EPISODES_PER_ROUND} episodes of {STEPS_PER_EPISODE} steps")
print(f"{MEASURE_NAME}: {before:.2f} before training, {after:.2f} after")
print(f"Learning curve:  {history[0]['measure']:.2f} in round 1, {history_table['measure'].tail(5).mean():.2f} over the last 5 rounds")

## 9. Save to the website ✏️

- The **model** is saved to the project as a new version of `MODEL_NAME` (saving again adds version 2, 3, ...), with its training record: the experiment, the number of episodes and the learning curve. It appears on the project page under **Trained rat models**.
- An **ONNX copy** of the network is saved with it, so the model can also drive the rat **live on the website**. Open any experiment in this project, go to **Setup → Who controls the rat → Saved model**, pick `MODEL_NAME` and this version, click **Use this**, then **Run → Start**. Doors, stimuli, speed and the **Rat's eyes** camera all work as usual.
- The **trained rat's run** and the **episode** from step 8 are saved under **Review** in this experiment. Open the links to replay them in 3D.

In [ ]:
MODEL_NAME = "Dark seeker"  #@param {type:"string"}

version = project.save_model(
    MODEL_NAME, model,
    description=TASKS[TASK]["description"],
    notes=f"PPO from rat_model.py. {TASK}, start {START or 'random'}, {STEPS_PER_EPISODE} steps per episode. "
          f"{MEASURE_NAME}: {before:.2f} before, {after:.2f} after training.",
    training=env.log)
print("Model saved:", version)
if version.onnx_path:
    print("It can drive the rat live on the website: Setup > Who controls the rat > Saved model >", version.label)
else:
    print("No ONNX copy was saved (see the warning above), so this version runs in Python only.")

model_run.save()
print("Trained rat's run:", model_run.review_link())

episode.save()
print(f"Episode {episode.number} at step {AT_STEP}:", episode.review_link(AT_STEP))

## 10. Reuse the model in another experiment ✏️ (optional)

A saved model belongs to the **project**, so any experiment in the project can use it: for example a copy of this experiment with the door closed, a different light, or a treatment. The run is saved to that experiment's Review, labelled with the model and its version.

Type the name of another experiment in this project and run the cell. (In a new notebook you would run steps 1 to 3 first; `rat_model.py` must define the same network that was saved.)

What carries over: the model reacts to what it senses (walls, light, sound, odour) and to where it is. A model trained in one arena layout behaves sensibly in experiments with the same layout; in a very different arena it is a starting point to train further, not a finished model.

In [ ]:
OTHER_EXPERIMENT = ""  #@param {type:"string"}

if OTHER_EXPERIMENT:
    other = project.experiment(OTHER_EXPERIMENT)
    saved = project.model(MODEL_NAME).latest                 # or .version(1), .version(2), ...
    reloaded = saved.load_into(RatModel())                   # an empty RatModel filled with the saved weights
    other_run = other.run(steps=2 * STEPS_PER_EPISODE, model=reloaded,
                          start=start.get("position"), episode_length=STEPS_PER_EPISODE)
    other_run.plot()
    plt.show()
    other_run.save()
    print(f"{saved} in {other.name}:", other_run.review_link())
else:
    print("Set OTHER_EXPERIMENT to try the model in another experiment of this project.")

## 11. Download your work (.pt and .onnx)

In [ ]:
import shutil
from google.colab import files

# Save PyTorch weights (.pt).
torch.save(model.state_dict(), "rat_model_weights.pt")

# Export the trained actor to ONNX (.onnx).
# Input: a batch of 31-number observations. Output: scores for the 5 discrete moves.
model.eval()
dummy_observation = torch.zeros(1, env.obs_size, dtype=torch.float32)
torch.onnx.export(
    model,
    dummy_observation,
    "rat_model.onnx",
    input_names=["observations"],
    output_names=["move_scores"],
    dynamic_axes={
        "observations": {0: "batch_size"},
        "move_scores": {0: "batch_size"},
    },
    opset_version=17,
    dynamo=False,
)
onnx.checker.check_model(onnx.load("rat_model.onnx"))
print("Exported and validated rat_model.onnx")

history_table.to_csv("training_history.csv", index=False)
shutil.make_archive("rodent-training", "zip", "rodent-training")

for name in [
    "rat_model.py",
    "rat_model_weights.pt",
    "rat_model.onnx",
    "training_history.csv",
    "rodent-training.zip",
]:
    files.download(name)


## Something went wrong?

- **Login failed:** check the two secrets (🔑 key icon) and that Notebook access is on for both.
- **No region called ...:** use a region name printed in step 4.
- **... needs an odour/sound source that is enabled and active:** see the tips in step 5.
- **The start position is inside a wall:** choose another `START`, or leave it empty for random starts.
- **The measure does not improve:** run step 7's training cell again to train longer; check in step 4 that the rat can sense what the task needs from where it starts; for `reach_region` in arenas with many walls, a nearer `GOAL` first works better than a far one.
- **... has unsaved changes:** the experiment was changed but not saved. Save it on the website and run the notebook again.
- **Saved without an ONNX copy:** the model is saved, but cannot drive the rat on the website. If the warning says `onnx` is missing, run `%pip install onnx` and step 9 again (it saves the next version). If it mentions the website's database, ask the project lead to apply the live-models migration.
- **The log was made with different settings:** each task keeps its own log folder; delete `rodent-training` (folder icon) to start a task's log again.

More in the [documentation](https://virtual-rodent-docs.pages.dev/).